# Step 7b — Quantization on GPU: vLLM fp16 vs AWQ 4-bit (Kaggle T4)

Part of **llm-serving-at-scale**. For each model variant this notebook measures, on the same GPU in the same session:
1. **quality** — the same 1,000 ARC-Easy questions used on CPU (Step 7a),
2. **speed** — the batch-size sweep from Step 6,
3. **capacity** — the 1000-user Locust load test,
4. **memory** — weight size and KV-cache size from vLLM's startup log.

Then it compares AWQ against fp16 *and* GPU fp16 against CPU fp16 (does the hardware/numeric path change answers?).

**Settings (right sidebar):** Accelerator **GPU T4 x2** · Internet **On** → **Run All**.
About 15 min with `QUALITY_ONLY = True` (quality re-run), ~30 min for everything.

**Decoding is pinned:** `--generation-config vllm` = plain greedy, like llama.cpp. (A first run without it
showed vLLM silently applying the model's repetition penalty 1.1, which penalizes the option letters
A-D that appear in the prompt and pushed the model into step-by-step answers.)
**Requires** the Step 7a files (`eval/`) to be pushed to GitHub — the notebook clones the repo.

In [ ]:
# ---- configuration -------------------------------------------------------------
REPO_URL = "https://github.com/rohithram22/llm-serving-at-scale.git"
VLLM_VERSION = "0.18.1"          # worked in Step 6 on Kaggle T4
VARIANTS = [
    # label (used in all result files), Hugging Face model, extra vLLM flags
    {"label": "vllm_t4_fp16", "model": "Qwen/Qwen2.5-0.5B-Instruct",     "extra": ""},
    {"label": "vllm_t4_awq",  "model": "Qwen/Qwen2.5-0.5B-Instruct-AWQ", "extra": "--quantization awq"},
]
GPU_MEM_UTIL = 0.85
MAX_MODEL_LEN = 4096
MAX_NUM_SEQS = 256
SWEEP_LEVELS = [1, 2, 4, 8, 16, 32, 64, 128, 256]
LOAD_USERS = [1000]              # the saturating case from Step 6 = capacity
LOAD_DURATION_S = 180
QUALITY_CONCURRENCY = 64
QUALITY_ONLY = True              # True = only re-run the 1,000 questions (speed/load results already committed)
# Plain greedy decoding, like llama.cpp. Without this, vLLM silently adopts the model's
# generation_config.json (repetition_penalty 1.1 ...), which penalizes the option letters in the prompt.
GEN_CONFIG = "--generation-config vllm"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/llm-serving-at-scale"
PORT = 8000

## 1. Check the GPU

In [ ]:
import subprocess

def sh(cmd: str, check: bool = True) -> str:
    """Run a shell command, print and return its output."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (p.stdout + p.stderr).strip()
    if out:
        print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return out

gpus = sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv,noheader")
cap = float(gpus.splitlines()[0].split(",")[-1])
assert cap >= 7.5, f"compute capability {cap} < 7.5: switch the accelerator to 'GPU T4 x2'."
print(f"\nOK: compute capability {cap}. Using GPU 0 only.")

## 2. Install vLLM + Locust, clone the repo (harness + eval set)

In [ ]:
import os, sys
spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
sh(f"{sys.executable} -m pip install -q {spec} locust 2>&1 | tail -3", check=False)
sh(f"{sys.executable} -m pip show vllm 2>/dev/null | grep -E '^(Name|Version)'", check=False)
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
os.makedirs("results/raw", exist_ok=True)
assert os.path.exists("eval/arc_easy_1000.jsonl") and os.path.exists("eval/mc_eval.py"), \
    "eval/ files not found in the repo: commit and push Step 7a first."
print("working in", os.getcwd())

## 3. vLLM start/stop helpers

Same as Step 6, including the **Kaggle fix** (FlashInfer needs a `libcuda.so` shortcut) and the automatic
**fallback to Triton attention** if the default backend fails. A crash is detected within ~20 s.

In [ ]:
import glob, json, time, urllib.request

LOG = f"{WORK}/vllm.log"

real = next((p for p in sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so.1") +
                               glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so.1") +
                               glob.glob("/usr/lib64/libcuda.so.1"))), None)
if real:
    shim_dir = f"{WORK}/cuda_link_shim"
    os.makedirs(shim_dir, exist_ok=True)
    if not os.path.lexists(f"{shim_dir}/libcuda.so"):
        os.symlink(real, f"{shim_dir}/libcuda.so")
    os.environ["LIBRARY_PATH"] = shim_dir + (":" + os.environ["LIBRARY_PATH"] if os.environ.get("LIBRARY_PATH") else "")
    print(f"linker shim: {shim_dir}/libcuda.so -> {real}")

def _port_answers() -> bool:
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2)
        return True
    except Exception:
        return False

def stop_vllm(timeout_s: int = 60):
    """Stop vLLM and WAIT until it is really gone (process exited, port free). Otherwise the next
    start can hit the OLD server's health check and send requests to the wrong model."""
    # '[v]llm serve' matches "vllm serve" but not this shell's own command line
    subprocess.run("pkill -f '[v]llm serve'", shell=True)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        alive = subprocess.run("pgrep -f '[v]llm serve'", shell=True, capture_output=True).returncode == 0
        if not alive and not _port_answers():
            return
        time.sleep(2)
    subprocess.run("pkill -9 -f '[v]llm serve'; sleep 3", shell=True)   # still there: force it

def serving_model() -> str | None:
    try:
        data = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=2).read())
        return data["data"][0]["id"]
    except Exception:
        return None

def wait_health(expected_model: str, timeout_s: int = 900) -> float:
    """Ready = /health is OK *and* /v1/models reports the model we just started."""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            if (urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2).status == 200
                    and serving_model() == expected_model):
                return time.time() - t0
        except Exception:
            pass
        if time.time() - t0 > 20 and subprocess.run("pgrep -f '[v]llm serve'", shell=True,
                                                     capture_output=True).returncode != 0:
            break                                   # process is gone -> it crashed
        time.sleep(3)
    print("---- vLLM failed to start. Engine errors: ----")
    sh(f"grep -v 'APIServer' {LOG} | grep -iE 'error|cannot|failed|not support' | tail -15", check=False)
    raise TimeoutError("vLLM did not become healthy")

def start_vllm(model: str, extra: str = "", timeout_s: int = 900) -> float:
    stop_vllm()
    cmd = (f"CUDA_VISIBLE_DEVICES=0 nohup vllm serve {model} --dtype half --port {PORT} "
           f"--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization {GPU_MEM_UTIL} "
           f"--max-num-seqs {MAX_NUM_SEQS} {GEN_CONFIG} {extra} > {LOG} 2>&1 &")
    subprocess.run(cmd, shell=True)
    secs = wait_health(model, timeout_s)
    print(f"vLLM ready after {secs:.0f}s | {model} {extra} | attention: "
          f"{os.environ.get('VLLM_ATTENTION_BACKEND', 'default (auto)')}")
    return secs

def start_with_fallback(model: str, extra: str = "") -> float:
    try:
        return start_vllm(model, extra)
    except TimeoutError:
        if os.environ.get("VLLM_ATTENTION_BACKEND") == "TRITON_ATTN":
            raise
        print("\n>>> Default attention backend failed. Plan B: Triton attention.")
        os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
        return start_vllm(model, extra)

## 4. Measurement helpers: batch sweep, startup facts, vLLM counters

In [ ]:
import asyncio, json, re
import httpx
import pandas as pd

PROMPT = "Explain what an API is to a beginner."

async def _one(client, model):
    t0 = time.perf_counter()
    r = await client.post(f"http://localhost:{PORT}/v1/chat/completions", json={
        "model": model, "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": 32, "temperature": 0, "ignore_eos": True})
    r.raise_for_status()
    return time.perf_counter() - t0, r.json()["usage"]["completion_tokens"]

async def sweep(model, levels):
    rows = []
    limits = httpx.Limits(max_connections=1024, max_keepalive_connections=1024)
    async with httpx.AsyncClient(timeout=300, limits=limits) as client:
        await asyncio.gather(*[_one(client, model) for _ in range(8)])          # warm-up
        for n in levels:
            t0 = time.perf_counter()
            res = await asyncio.gather(*[_one(client, model) for _ in range(n)])
            wall = time.perf_counter() - t0
            lat = [x[0] for x in res]
            rows.append({"concurrency": n, "wall_s": round(wall, 3), "req_per_s": round(n / wall, 2),
                         "tokens_per_s": round(sum(x[1] for x in res) / wall, 1),
                         "mean_latency_s": round(sum(lat) / n, 3),
                         "ms_per_step": round(1000 * (sum(lat) / n) / 32, 1)})
    df = pd.DataFrame(rows)
    df["speedup_vs_1"] = (df["tokens_per_s"] / df.loc[0, "tokens_per_s"]).round(1)
    return df

def startup_facts(log_text: str) -> dict:
    def grab(pattern, cast=float):
        m = re.search(pattern, log_text)
        return cast(m.group(1).replace(",", "")) if m else None
    return {"weights_gib": grab(r"Model loading took ([\d.]+) GiB"),
            "kv_cache_gib": grab(r"Available KV cache memory: ([\d.]+) GiB"),
            "kv_cache_tokens": grab(r"GPU KV cache size: ([\d,]+) tokens", lambda s: int(float(s))),
            "attention_backend": (re.search(r"Using (\w+) attention backend", log_text) or [None, None])[1],
            "quant_method": (re.search(r"quantization=(\w+)", log_text) or [None, None])[1],
            # must be False: True means the model's own sampling defaults (repetition penalty) are active
            "hf_sampling_defaults_applied": "overridden by the model's" in log_text}

def vllm_counters(path):
    keep = ("num_requests_waiting", "prefix_cache_queries", "prefix_cache_hits", "num_preemptions")
    vals = {}
    for line in open(path):
        if line.startswith("#"):
            continue
        m = re.match(r"vllm:([a-z_]+)(?:\{[^}]*\})?\s+([0-9.eE+-]+)$", line.strip())
        if m and any(k in m.group(1) for k in keep):
            vals[m.group(1)] = vals.get(m.group(1), 0.0) + float(m.group(2))
    return vals

## 5. Run every variant: start → quality → sweep → load test

For each variant: a fresh server, then the 1,000 questions (same eval script as on CPU), then the sweep,
then the load test(s) on a fresh server each. Takes ~10 min per variant.

In [ ]:
facts, sweeps = {}, {}
for v in VARIANTS:
    label, model, extra = v["label"], v["model"], v["extra"]
    print(f"\n{'=' * 20} {label} {'=' * 20}")
    start_with_fallback(model, extra)
    sh(f"cp {LOG} results/raw/{label}_startup.log", check=False)
    facts[label] = startup_facts(open(LOG).read())
    print("startup facts:", facts[label])

    # quality: same 1,000 questions, same prompt and parser as CPU (Step 7a)
    sh(f"python eval/mc_eval.py --label {label} --url http://localhost:{PORT} "
       f"--model {model} --concurrency {QUALITY_CONCURRENCY}", check=True)    # fail loudly

    assert not facts[label]["hf_sampling_defaults_applied"], "vLLM is still using the model's sampling defaults"

    if QUALITY_ONLY:                                  # reuse the committed speed results for the summary
        path = f"results/{label}_batch_sweep.csv"
        if os.path.exists(path):
            sweeps[label] = pd.read_csv(path)
        continue

    # speed
    sweeps[label] = await sweep(model, SWEEP_LEVELS)
    sweeps[label].to_csv(f"results/{label}_batch_sweep.csv", index=False)
    print(sweeps[label][["concurrency", "tokens_per_s", "mean_latency_s", "ms_per_step"]].to_string(index=False))

    # capacity (fresh server per run)
    for users in LOAD_USERS:
        start_with_fallback(model, extra)
        sh(f"MODEL_NAME={model} bash loadtest/run_load.sh {label} {users} {LOAD_DURATION_S} "
           f"http://localhost:{PORT}", check=False)
        mfile = f"results/raw/{label}_u{users}_metrics.txt"
        if os.path.exists(mfile) and os.path.getsize(mfile):
            c = vllm_counters(mfile)
            q = c.get("prefix_cache_queries_total", 0)
            print(f"vLLM counters @ {users} users: preemptions={c.get('num_preemptions_total')} "
                  f"prefix-cache hit rate={c.get('prefix_cache_hits_total', 0) / q:.0%}" if q else c)
stop_vllm()

## 6. Quality comparisons (paired, same questions)

In [ ]:
labels = [v["label"] for v in VARIANTS]
print("---- AWQ vs fp16, both on the GPU ----")
_ = sh(f"python eval/compare_quality.py {' '.join(labels)}", check=False)
if os.path.exists("results/quality/llamacpp_fp16_preds.csv"):
    print("\n---- same fp16 weights, different hardware: CPU (llama.cpp) vs GPU (vLLM) ----")
    _ = sh("python eval/compare_quality.py llamacpp_fp16 vllm_t4_fp16", check=False)
    # compare_quality names its output after the baseline; rename so it can't overwrite the
    # Step 7a CPU file (quality_comparison_llamacpp_fp16.csv) when the results are unzipped.
    os.replace("results/quality_comparison_llamacpp_fp16.csv", "results/quality_comparison_cpu_vs_gpu_fp16.csv")
    print("saved results/quality_comparison_cpu_vs_gpu_fp16.csv")

## 7. Chart: fp16 vs AWQ batch sweep on the T4

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

BLUE, AQUA = "#2a78d6", "#1baf7a"                     # categorical slots: fp16, AWQ
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e7e6e2", "#fcfcfb"
names = {"vllm_t4_fp16": "fp16", "vllm_t4_awq": "AWQ 4-bit"}
colors = {"vllm_t4_fp16": BLUE, "vllm_t4_awq": AQUA}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), facecolor=SURFACE)
for ax, (col, title) in zip(axes, [("tokens_per_s", "Throughput (output tokens/s)"),
                                   ("mean_latency_s", "Latency per request (s)")]):
    ax.set_facecolor(SURFACE)
    for label, df in sweeps.items():
        ax.plot(df["concurrency"], df[col], color=colors.get(label, INK2), lw=2, marker="o", ms=8,
                markeredgecolor=SURFACE, markeredgewidth=2, label=names.get(label, label), zorder=3)
        last = df.iloc[-1]
        ax.annotate(f"{last[col]:,.0f}" if col == "tokens_per_s" else f"{last[col]:.2f}s",
                    (last["concurrency"], last[col]), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK2)
    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_locator(FixedLocator(SWEEP_LEVELS))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v)}"))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("concurrent requests (= batch size)", color=INK2, fontsize=9)
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.tick_params(colors=INK2, labelsize=9)
    for s in ax.spines.values():
        s.set_visible(False)
axes[0].legend(frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle("vLLM on a T4: fp16 vs AWQ 4-bit (same model, same prompts)", x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout()
fig.savefig("results/quant_gpu_fp16_vs_awq.png", dpi=150, facecolor=SURFACE)
plt.show()

## 8. One summary table (size · memory · speed · capacity · quality)

In [ ]:
qs = pd.read_csv("results/quality_summary.csv")
cmp_path = f"results/quality_comparison_{VARIANTS[0]['label']}.csv"
cmp = pd.read_csv(cmp_path) if os.path.exists(cmp_path) else pd.DataFrame()
load = pd.read_csv("results/loadtest_summary.csv")
rows = []
for v in VARIANTS:
    label = v["label"]
    sw = sweeps[label].set_index("concurrency") if label in sweeps else pd.DataFrame()
    pick = lambda conc, col: sw.loc[conc, col] if conc in sw.index else None
    q = qs[qs["label"] == label].iloc[-1]
    ld = load[(load["label"] == label) & (load["users"] == LOAD_USERS[-1])]
    c = cmp[cmp["label"] == label] if len(cmp) else pd.DataFrame()
    rows.append({"variant": label, **facts[label],
                 "tok_s_batch1": pick(1, "tokens_per_s"), "tok_s_batch64": pick(64, "tokens_per_s"),
                 "ms_step_batch1": pick(1, "ms_per_step"), "ms_step_batch64": pick(64, "ms_per_step"),
                 f"goodput_{LOAD_USERS[-1]}u": ld.iloc[-1]["goodput_rps"] if len(ld) else None,
                 f"p95_{LOAD_USERS[-1]}u_s": ld.iloc[-1]["p95_s"] if len(ld) else None,
                 "accuracy": q["accuracy"], "unparsed (not in answer format)": q["unparsed"],
                 "agree_vs_fp16": c.iloc[-1]["agreement"] if len(c) else None,
                 "mcnemar_p": c.iloc[-1]["mcnemar_p"] if len(c) else None})
summary = pd.DataFrame(rows)
summary.to_csv("results/quant_gpu_summary.csv", index=False)
pd.set_option("display.width", 250)
print(summary.T.to_string(header=False))

## 9. Save results → download `step7b_results.zip` from the Output panel

In [ ]:
import zipfile
env = sh(f"nvidia-smi --query-gpu=name,driver_version --format=csv,noheader | head -1; "
         f"{sys.executable} -m pip show vllm torch 2>/dev/null | grep -E '^(Name|Version)'", check=False)
with open("results/quant_gpu_environment.txt", "w") as f:
    f.write(env + f"\nattention backend setting: {os.environ.get('VLLM_ATTENTION_BACKEND', 'default (auto)')}\n")
files = (["results/loadtest_summary.csv", "results/quality_summary.csv", "results/quant_gpu_summary.csv",
          "results/quant_gpu_fp16_vs_awq.png", "results/quant_gpu_environment.txt"]
         + [f"results/quality_comparison_{VARIANTS[0]['label']}.csv", "results/quality_comparison_cpu_vs_gpu_fp16.csv"]
         + [f"results/quality/{v['label']}_preds.csv" for v in VARIANTS]
         + [f"results/{v['label']}_batch_sweep.csv" for v in VARIANTS])
zpath = f"{WORK}/step7b_results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for p in files:
        if os.path.exists(p):
            z.write(p)
print(f"saved {zpath}:", *[p for p in files if os.path.exists(p)], sep="\n  ")

# Kaggle's Output panel doesn't always offer "Download" -> a direct link (paths are relative to /kaggle/working)
from IPython.display import FileLink, display
os.chdir(WORK)
display(FileLink("step7b_results.zip"))